## 1. Estado Objetivo e Exibição do Tabuleiro

In [49]:
OBJETIVO = (1, 2, 3, 4, 5, 6, 7, 8, 'X')  # Define a tupla com o estado final almejado

def imprimir_tabuleiro(estado):  # Função para exibir o tabuleiro 3x3 no console
    """Imprime o tabuleiro 3x3 no formato solicitado pelo professor."""
    print(f"|{estado[0]} | {estado[1]} | {estado[2]}|")  # Imprime a primeira linha do tabuleiro
    print(f"|{estado[3]} | {estado[4]} | {estado[5]}|")  # Imprime a segunda linha do tabuleiro
    print(f"|{estado[6]} | {estado[7]} | {estado[8]}|")  # Imprime a terceira linha do tabuleiro

print("Estado Objetivo:")  # Exibe o título do estado objetivo
imprimir_tabuleiro(OBJETIVO)  # Chama a função para imprimir o estado objetivo


Estado Objetivo:
|1 | 2 | 3|
|4 | 5 | 6|
|7 | 8 | X|


## 2. Verificação de Solubilidade


In [50]:
def contar_inversoes(estado):  # Função que calcula o número de inversões numéricas
    """Conta quantos números maiores aparecem antes de números menores."""
    numeros = [x for x in estado if x != 'X']  # Extrai apenas números, ignorando o espaço vazio 'X'
    inversoes = 0  # Inicializa o contador de inversões com zero
    for i in range(len(numeros)):  # Percorre cada elemento da lista
        for j in range(i + 1, len(numeros)):  # Compara com todos os elementos posteriores
            if numeros[i] > numeros[j]:  # Se o número anterior for maior que o posterior
                inversoes += 1  # Incrementa o total de inversões
    return inversoes  # Retorna a contagem total de inversões

def eh_soluvel(estado):  # Função que verifica a solubilidade do tabuleiro 3x3
    """Retorna True se o número de inversões for par."""
    return contar_inversoes(estado) % 2 == 0  # É solúvel se o número de inversões for par

tabuleiro_par = (1, 2, 3, 4, 5, 6, 7, 'X', 8)  # Tabuleiro de exemplo com inversões pares
tabuleiro_impar = (2, 1, 3, 4, 5, 6, 7, 8, 'X')  # Tabuleiro de exemplo com inversões ímpares

print(f"Teste 1: {contar_inversoes(tabuleiro_par)} inversões -> Solúvel? {eh_soluvel(tabuleiro_par)}")  # Exibe teste do tabuleiro par
print(f"Teste 2: {contar_inversoes(tabuleiro_impar)} inversão -> Solúvel? {eh_soluvel(tabuleiro_impar)}")  # Exibe teste do tabuleiro ímpar


Teste 1: 0 inversões -> Solúvel? True
Teste 2: 1 inversão -> Solúvel? False


## 3. Geração do Estado Inicial Aleatório


In [51]:
import random  # Biblioteca para operações aleatórias

def gerar_estado_inicial():  # Função para gerar um tabuleiro inicial solúvel
    """Gera um estado inicial aleatório e garante que ele seja solúvel."""
    pecas = [1, 2, 3, 4, 5, 6, 7, 8, 'X']  # Lista com todas as peças do jogo
    tentativa = 1  # Contador de tentativas de geração
    while True:  # Loop contínuo até encontrar configuração válida
        random.shuffle(pecas)  # Embaralha as peças aleatoriamente
        estado = tuple(pecas)  # Converte a lista embaralhada em tupla imutável
        inversoes = contar_inversoes(estado)  # Calcula as inversões da configuração
        
        if eh_soluvel(estado) and estado != OBJETIVO:  # Verifica se tem solução e não é o objetivo
            print(f"Tentativa {tentativa}: Estado solúvel aceito! (Inversões: {inversoes} - Par)")  # Mensagem de sucesso
            return estado  # Retorna o tabuleiro válido encontrado
        else:  # Se for ímpar (insolúvel) ou já igual ao objetivo
            print(f"Tentativa {tentativa}: Descartado (Inversões: {inversoes} - Ímpar, gerando outro...)")  # Informa descarte
            tentativa += 1  # Incrementa o contador para a próxima tentativa

random.seed(15)  # Fixa a semente pseudoaleatória para reprodutibilidade

estado_inicial = gerar_estado_inicial()  # Gera o estado inicial solúvel
print("\nEstado Inicial:")  # Exibe rótulo do estado gerado
imprimir_tabuleiro(estado_inicial)  # Imprime o tabuleiro inicial no formato 3x3


Tentativa 1: Descartado (Inversões: 17 - Ímpar, gerando outro...)
Tentativa 2: Estado solúvel aceito! (Inversões: 12 - Par)

Estado Inicial:
|8 | 2 | 4|
|3 | 1 | 6|
|5 | 7 | X|


## 4. Movimentos Válidos (Sucessores)

In [52]:
def gerar_sucessores(estado):  # Função que gera todos os movimentos válidos a partir do estado atual
    """Gera os próximos estados possíveis movimentando o 'X'."""
    sucessores = []  # Lista para armazenar os novos estados gerados
    idx = estado.index('X')  # Localiza o índice da peça vazia 'X' (0 a 8)
    linha = idx // 3  # Calcula a linha atual de 'X' (divisão inteira por 3)
    coluna = idx % 3  # Calcula a coluna atual de 'X' (resto da divisão por 3)

    movimentos = [  # Lista de deslocamentos possíveis (delta linha, delta coluna)
        (-1, 0),  # Cima: sobe uma linha
        (1, 0),   # Baixo: desce uma linha
        (0, -1),  # Esquerda: volta uma coluna
        (0, 1)    # Direita: avança uma coluna
    ]

    for dl, dc in movimentos:  # Itera sobre cada direção de movimento
        nova_l = linha + dl  # Calcula a nova posição da linha
        nova_c = coluna + dc  # Calcula a nova posição da coluna
        
        if 0 <= nova_l < 3 and 0 <= nova_c < 3:  # Verifica se a nova posição está dentro do tabuleiro 3x3
            novo_idx = nova_l * 3 + nova_c  # Converte coordenadas (linha, coluna) de volta para índice 1D
            novo_estado = list(estado)  # Converte a tupla para lista para permitir modificação
            novo_estado[idx], novo_estado[novo_idx] = novo_estado[novo_idx], novo_estado[idx]  # Troca 'X' de lugar com o vizinho
            sucessores.append(tuple(novo_estado))  # Converte de volta para tupla e adiciona aos sucessores

    return sucessores  # Retorna a lista de todos os estados válidos seguintes


## 5. Busca em Largura (BFS)


In [53]:
from collections import deque  # Importa deque para fila FIFO eficiente O(1)

def busca_largura(estado_inicial, max_impressoes=10):  # Função que executa a Busca em Largura (BFS)
    """Executa a Busca em Largura (BFS)."""
    print("=" * 45)  # Imprime linha separadora visual
    print("BUSCA EM LARGURA (BFS)")  # Imprime o cabeçalho do algoritmo
    print("=" * 45)  # Imprime linha separadora visual
    print("Estado inicial:")  # Exibe título do estado inicial
    imprimir_tabuleiro(estado_inicial)  # Mostra o tabuleiro inicial formatado
    print()  # Pula uma linha no console

    fila = deque([estado_inicial])  # Fila FIFO contendo inicialmente apenas o estado inicial
    visitados = {estado_inicial}  # Conjunto de estados já visitados para evitar ciclos
    iteracoes = 0  # Contador de nós expandidos/iterações realizadas

    while fila:  # Enquanto houver nós para explorar na fila
        iteracoes += 1  # Incrementa a contagem de iterações
        atual = fila.popleft()  # Retira o nó mais antigo da fila (ordem FIFO)

        if iteracoes <= max_impressoes:  # Se estiver dentro do limite de impressões
            print(f"Iteração {iteracoes}:")  # Imprime o número da iteração atual
            imprimir_tabuleiro(atual)  # Exibe o tabuleiro atual
            print()  # Pula linha
        elif iteracoes == max_impressoes + 1:  # Ao ultrapassar o limite de impressões
            print("...\n")  # Imprime reticências para indicar continuação sem poluir saída

        if atual == OBJETIVO:  # Verifica se o estado atual é a meta desejada
            print("Estado Final:")  # Informa que o objetivo foi atingido
            imprimir_tabuleiro(atual)  # Imprime o tabuleiro final
            print(f"\nNúmero de iterações para busca em largura N: {iteracoes}")  # Mostra total de iterações
            return iteracoes  # Retorna o número de nós explorados até o objetivo

        for vizinho in gerar_sucessores(atual):  # Gera todos os movimentos possíveis a partir do estado atual
            if vizinho not in visitados:  # Se o estado ainda não foi visitado anteriormente
                visitados.add(vizinho)  # Adiciona ao conjunto de visitados
                fila.append(vizinho)  # Insere no final da fila para processamento posterior

    return iteracoes  # Retorna iterações caso termine sem encontrar solução

n_bfs = busca_largura(estado_inicial)  # Executa o BFS e armazena o número total de iterações


BUSCA EM LARGURA (BFS)
Estado inicial:
|8 | 2 | 4|
|3 | 1 | 6|
|5 | 7 | X|

Iteração 1:
|8 | 2 | 4|
|3 | 1 | 6|
|5 | 7 | X|

Iteração 2:
|8 | 2 | 4|
|3 | 1 | X|
|5 | 7 | 6|

Iteração 3:
|8 | 2 | 4|
|3 | 1 | 6|
|5 | X | 7|

Iteração 4:
|8 | 2 | X|
|3 | 1 | 4|
|5 | 7 | 6|

Iteração 5:
|8 | 2 | 4|
|3 | X | 1|
|5 | 7 | 6|

Iteração 6:
|8 | 2 | 4|
|3 | X | 6|
|5 | 1 | 7|

Iteração 7:
|8 | 2 | 4|
|3 | 1 | 6|
|X | 5 | 7|

Iteração 8:
|8 | X | 2|
|3 | 1 | 4|
|5 | 7 | 6|

Iteração 9:
|8 | X | 4|
|3 | 2 | 1|
|5 | 7 | 6|

Iteração 10:
|8 | 2 | 4|
|3 | 7 | 1|
|5 | X | 6|

...

Estado Final:
|1 | 2 | 3|
|4 | 5 | 6|
|7 | 8 | X|

Número de iterações para busca em largura N: 80950


## 6. Busca em Profundidade (DFS)


In [54]:
def busca_profundidade(estado_inicial, max_impressoes=10):  # Função que executa a Busca em Profundidade (DFS)
    """Executa a Busca em Profundidade (DFS)."""
    print("=" * 45)  # Imprime linha separadora visual
    print("BUSCA EM PROFUNDIDADE (DFS)")  # Imprime o cabeçalho do algoritmo
    print("=" * 45)  # Imprime linha separadora visual
    print("Estado inicial:")  # Exibe título do estado de partida
    imprimir_tabuleiro(estado_inicial)  # Mostra o tabuleiro inicial formatado
    print()  # Pula uma linha no console

    pilha = [estado_inicial]  # Pilha LIFO contendo o estado de partida
    visitados = {estado_inicial}  # Conjunto de visitados para evitar loops infinitos
    iteracoes = 0  # Contador de nós explorados/iterações

    while pilha:  # Enquanto houver nós na pilha
        iteracoes += 1  # Incrementa a contagem de iterações
        atual = pilha.pop()  # Retira o elemento do topo da pilha (ordem LIFO)

        if iteracoes <= max_impressoes:  # Se estiver dentro do limite de impressões
            print(f"Iteração {iteracoes}:")  # Imprime o número da iteração atual
            imprimir_tabuleiro(atual)  # Exibe o tabuleiro atual
            print()  # Pula linha
        elif iteracoes == max_impressoes + 1:  # Ao ultrapassar o limite de impressões
            print("...\n")  # Imprime reticências indicando continuação silenciosa

        if atual == OBJETIVO:  # Verifica se o estado atual alcançou o objetivo
            print("Estado Final:")  # Informa conclusão do objetivo
            imprimir_tabuleiro(atual)  # Imprime o tabuleiro final
            print(f"\nNúmero de iterações para busca em profundidade N: {iteracoes}")  # Mostra total de iterações
            return iteracoes  # Retorna o total de iterações realizadas até a meta

        for vizinho in reversed(gerar_sucessores(atual)):  # Percorre sucessores em ordem reversa para desempilhar na ordem natural
            if vizinho not in visitados:  # Se o vizinho ainda não foi visitado
                visitados.add(vizinho)  # Marca o vizinho como visitado
                pilha.append(vizinho)  # Adiciona ao topo da pilha para ser explorado a seguir

    return iteracoes  # Retorna iterações caso a pilha se esvazie

n_dfs = busca_profundidade(estado_inicial)  # Executa o DFS e armazena o número total de iterações


BUSCA EM PROFUNDIDADE (DFS)
Estado inicial:
|8 | 2 | 4|
|3 | 1 | 6|
|5 | 7 | X|

Iteração 1:
|8 | 2 | 4|
|3 | 1 | 6|
|5 | 7 | X|

Iteração 2:
|8 | 2 | 4|
|3 | 1 | X|
|5 | 7 | 6|

Iteração 3:
|8 | 2 | X|
|3 | 1 | 4|
|5 | 7 | 6|

Iteração 4:
|8 | X | 2|
|3 | 1 | 4|
|5 | 7 | 6|

Iteração 5:
|8 | 1 | 2|
|3 | X | 4|
|5 | 7 | 6|

Iteração 6:
|8 | 1 | 2|
|3 | 7 | 4|
|5 | X | 6|

Iteração 7:
|8 | 1 | 2|
|3 | 7 | 4|
|X | 5 | 6|

Iteração 8:
|8 | 1 | 2|
|X | 7 | 4|
|3 | 5 | 6|

Iteração 9:
|X | 1 | 2|
|8 | 7 | 4|
|3 | 5 | 6|

Iteração 10:
|1 | X | 2|
|8 | 7 | 4|
|3 | 5 | 6|

...

Estado Final:
|1 | 2 | 3|
|4 | 5 | 6|
|7 | 8 | X|

Número de iterações para busca em profundidade N: 24098


## 7. Resumo Final das Iterações

In [55]:
print("=" * 45)  # Imprime linha separadora visual
print("RESUMO FINAL:")  # Imprime título da seção de resumo
print(f"Número de iterações para busca em largura N: {n_bfs}")  # Exibe o total de nós na busca em largura
print(f"Número de iterações para busca em profundidade N: {n_dfs}")  # Exibe o total de nós na busca em profundidade
print("=" * 45)  # Imprime linha separadora de encerramento


RESUMO FINAL:
Número de iterações para busca em largura N: 80950
Número de iterações para busca em profundidade N: 24098


### Demonstração

In [56]:
tabuleiro_curto = (1, 2, 3, 4, 'X', 6, 7, 5, 8)  # Configuração inicial próxima da meta para teste rápido

print("--- Teste Rápido com BFS ---")  # Rótulo de teste para Busca em Largura
busca_largura(tabuleiro_curto, max_impressoes=20)  # Executa BFS no tabuleiro teste exibindo até 20 passos

print("\n--- Teste Rápido com DFS ---")  # Rótulo de teste para Busca em Profundidade
busca_profundidade(tabuleiro_curto, max_impressoes=20)  # Executa DFS no tabuleiro teste exibindo até 20 passos


--- Teste Rápido com BFS ---
BUSCA EM LARGURA (BFS)
Estado inicial:
|1 | 2 | 3|
|4 | X | 6|
|7 | 5 | 8|

Iteração 1:
|1 | 2 | 3|
|4 | X | 6|
|7 | 5 | 8|

Iteração 2:
|1 | X | 3|
|4 | 2 | 6|
|7 | 5 | 8|

Iteração 3:
|1 | 2 | 3|
|4 | 5 | 6|
|7 | X | 8|

Iteração 4:
|1 | 2 | 3|
|X | 4 | 6|
|7 | 5 | 8|

Iteração 5:
|1 | 2 | 3|
|4 | 6 | X|
|7 | 5 | 8|

Iteração 6:
|X | 1 | 3|
|4 | 2 | 6|
|7 | 5 | 8|

Iteração 7:
|1 | 3 | X|
|4 | 2 | 6|
|7 | 5 | 8|

Iteração 8:
|1 | 2 | 3|
|4 | 5 | 6|
|X | 7 | 8|

Iteração 9:
|1 | 2 | 3|
|4 | 5 | 6|
|7 | 8 | X|

Estado Final:
|1 | 2 | 3|
|4 | 5 | 6|
|7 | 8 | X|

Número de iterações para busca em largura N: 9

--- Teste Rápido com DFS ---
BUSCA EM PROFUNDIDADE (DFS)
Estado inicial:
|1 | 2 | 3|
|4 | X | 6|
|7 | 5 | 8|

Iteração 1:
|1 | 2 | 3|
|4 | X | 6|
|7 | 5 | 8|

Iteração 2:
|1 | X | 3|
|4 | 2 | 6|
|7 | 5 | 8|

Iteração 3:
|X | 1 | 3|
|4 | 2 | 6|
|7 | 5 | 8|

Iteração 4:
|4 | 1 | 3|
|X | 2 | 6|
|7 | 5 | 8|

Iteração 5:
|4 | 1 | 3|
|7 | 2 | 6|
|X | 5 | 8|



329